# Model Optimization Workshop: Introduction and Setup

Welcome to the Model Optimization Workshop! In this workshop, you'll learn how to optimize machine learning models to improve performance and reduce costs. This first notebook will guide you through setting up your environment and preparing for the subsequent notebooks.

# Part 1: Environment Setup

## 1. Install Required Packages

We'll install the basic packages needed for this notebook. Each subsequent notebook will install its specific dependencies as needed.

In [ ]:
# Install only the packages needed for this notebook
!pip install -q "boto3>=1.35.0" "sagemaker>=2.230.0" "pandas>=2.2.0" "numpy>=1.26.0" "torch>=2.4.0" "torchvision>=0.19.0" "transformers>=4.44.0"

## 2. Verify Environment

In [ ]:
# Import standard libraries
import os
import sys
import time
import json

# Data processing
import numpy as np
import pandas as pd

# Machine learning
import torch

# AWS
import boto3
import sagemaker

# Print versions of key packages
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Boto3 version: {boto3.__version__}")
print(f"SageMaker version: {sagemaker.__version__}")

# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")

## 3. Set Up SageMaker Session

In [ ]:
# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.session.Session().region_name
bucket = sagemaker_session.default_bucket()
prefix = "model-optimization-workshop"

print(f"SageMaker session established in region: {region}")
print(f"Using S3 bucket: {bucket}")
print(f"Using S3 prefix: {prefix}")

## 4. Understanding SageMaker Job Types

Before we dive into the workshop, it's important to understand the two main types of compute jobs you'll encounter in SageMaker and their corresponding APIs. This knowledge will help you understand why we use different approaches in different notebooks.

### Training Jobs vs Processing Jobs

SageMaker provides two primary job types for different use cases:

| Aspect | Training Jobs | Processing Jobs |
|--------|---------------|------------------|
| **Primary Purpose** | Train or fine-tune ML models | Data processing, model optimization, batch inference |
| **Input/Output** | Training data → Trained model | Any data → Processed data |
| **Built-in Features** | Automatic model artifacts handling, hyperparameter tuning, distributed training | Flexible input/output, custom processing logic |
| **Typical Use Cases** | Model training, fine-tuning, hyperparameter optimization | Data preprocessing, model quantization, batch predictions |
| **Workshop Examples** | Fine-tuning notebook (05) | Quantization notebook (02) |

### Estimators vs Processors

These job types are accessed through different API patterns:

#### Estimators (for Training Jobs)
- **High-level, framework-specific APIs**
- Examples: `HuggingFace`, `PyTorch`, `TensorFlow`, `SKLearn`
- **Built-in optimizations** for each framework
- **Automatic handling** of model artifacts and metrics
- **Easy deployment** with `.deploy()` method

#### Processors (for Processing Jobs)
- **Lower-level, more flexible APIs**
- Examples: `PyTorchProcessor`, `ScriptProcessor`, `FrameworkProcessor`
- **Custom processing logic** with your own scripts
- **Flexible input/output** configuration
- **No automatic model handling** - you control everything

### Code Pattern Comparison

Here's how the APIs differ in practice:

#### Training Job with Estimator (Fine-tuning)
```python
# High-level, framework-specific
estimator = HuggingFace(
    entry_point='train.py',
    instance_type='ml.g5.xlarge',
    transformers_version='4.44.0',
    pytorch_version='2.4.0',
    hyperparameters={'epochs': 3, 'learning_rate': 5e-5}
)

# Automatic model artifact handling
estimator.fit({'train': train_data, 'validation': val_data})

# Easy deployment
predictor = estimator.deploy(instance_type='ml.m5.xlarge')
```

#### Processing Job with Processor (Quantization)
```python
# Lower-level, more flexible
processor = PyTorchProcessor(
    framework_version='2.4.0',
    instance_type='ml.c5.xlarge',
    instance_count=1
)

# Manual input/output configuration
processor.run(
    code='quantization_script.py',
    inputs=[ProcessingInput(source=model_s3_uri, destination='/opt/ml/processing/input')],
    outputs=[ProcessingOutput(source='/opt/ml/processing/output', destination=output_s3_uri)],
    arguments=['--quantization-approach', 'dynamic']
)
```

### When to Use Each Approach

**Use Training Jobs + Estimators when:**
- Training or fine-tuning models
- You want framework-specific optimizations
- You need automatic model artifact management
- You plan to deploy the model directly
- You want built-in hyperparameter tuning

**Use Processing Jobs + Processors when:**
- Preprocessing data
- Optimizing existing models (quantization, pruning)
- Running batch inference
- Custom data transformations
- You need maximum flexibility in your processing logic

### Workshop Application

Throughout this workshop, you'll see both patterns:

- **Notebook 02 (Quantization)**: Uses `PyTorchProcessor` because we're optimizing an existing model
- **Notebook 05 (Fine-tuning)**: Uses `HuggingFace` estimator because we're training a model

Understanding these patterns will help you choose the right approach for your own ML workflows.

## 5. Create Workshop Configuration

Let's create a configuration file that will be used across all notebooks to ensure consistency.

In [ ]:
# Create workshop configuration
workshop_config = {
    "base_model": "distilbert-base-uncased",
    "task": "sequence-classification",
    "s3_bucket": bucket,
    "s3_prefix": prefix,
    "region": region,
    "role": role,
    "device_type": "GPU" if torch.cuda.is_available() else "CPU",
    "created_at": time.strftime("%Y-%m-%d-%H-%M-%S")
}

# Save configuration to file
with open("workshop_config.json", "w") as f:
    json.dump(workshop_config, f, indent=2)

print("Workshop configuration saved to workshop_config.json")

## 6. Store Variables for Other Notebooks

Let's create and store variables that other notebooks in the workshop will need. This ensures consistency across all notebooks.

In [ ]:
# Create variables with the names expected by other notebooks
S3_BUCKET = bucket
SAGEMAKER_ROLE_ARN = role
AWS_REGION = region
OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Instance type for processing jobs

# Store variables for other notebooks using IPython magic
%store S3_BUCKET
%store SAGEMAKER_ROLE_ARN
%store AWS_REGION
%store OPTIMIZATION_INSTANCE_TYPE

print("Variables stored for other notebooks:")
print(f"S3_BUCKET: {S3_BUCKET}")
print(f"SAGEMAKER_ROLE_ARN: {SAGEMAKER_ROLE_ARN}")
print(f"AWS_REGION: {AWS_REGION}")
print(f"OPTIMIZATION_INSTANCE_TYPE: {OPTIMIZATION_INSTANCE_TYPE}")

## 7. Create Model Information for Workshop

Let's create a model information file that other notebooks will use to know which models to work with.

In [ ]:
# Create model information that other notebooks expect
model_info = {
    "sentiment-analysis": {
        "model_name": "distilbert-base-uncased",
        "task": "text-classification",
        "hub_model_id": "distilbert-base-uncased",
        "s3_uri": f"s3://{bucket}/models/distilbert-base-uncased"
    }
}

# Save model info to file
with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print("Model information saved to model_info.json")
print(f"Available models: {list(model_info.keys())}")

## 8. Download and Upload Base Models to S3

Now we need to download the base models from Hugging Face Hub and upload them to S3 so that the other notebooks can use them as inputs for processing jobs.

In [ ]:
# Import additional libraries needed for model download and upload
import tempfile
import tarfile
import shutil
from transformers import AutoModel, AutoTokenizer

print("Starting model download and upload process...")

# Process each model in our model_info
for model_key, model_data in model_info.items():
    model_name = model_data["model_name"]
    s3_uri = model_data["s3_uri"]
    
    print(f"\nProcessing model: {model_name}")
    print(f"Target S3 URI: {s3_uri}")
    
    # Set up S3 client and paths
    s3_client = boto3.client('s3')
    bucket_name = s3_uri.split('/')[2]
    s3_key = '/'.join(s3_uri.split('/')[3:]) + '/model.tar.gz'
    
    # Create temporary directory for model files
    with tempfile.TemporaryDirectory() as temp_dir:
        model_dir = os.path.join(temp_dir, "model")
        os.makedirs(model_dir, exist_ok=True)
        
        print(f"Downloading model from Hugging Face Hub...")
        
        try:
            # Download model and tokenizer
            model = AutoModel.from_pretrained(model_name)
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            
            # Save model and tokenizer to local directory
            model.save_pretrained(model_dir)
            tokenizer.save_pretrained(model_dir)
            
            print(f"Model downloaded and saved locally")
            
            # Create model.tar.gz file
            tar_path = os.path.join(temp_dir, "model.tar.gz")
            print(f"Creating model.tar.gz...")
            
            with tarfile.open(tar_path, "w:gz") as tar:
                tar.add(model_dir, arcname=".")
            
            # Get file size for reporting
            file_size_mb = os.path.getsize(tar_path) / (1024 * 1024)
            print(f"Created model.tar.gz ({file_size_mb:.2f} MB)")
            
            # Upload to S3
            print(f"Uploading to S3...")
            s3_client.upload_file(tar_path, bucket_name, s3_key)
            
            print(f"✅ Successfully uploaded {model_name} to {s3_uri}/model.tar.gz")
            
        except Exception as e:
            print(f"❌ Error processing model {model_name}: {e}")
            raise

print("\n🎉 All models have been downloaded and uploaded to S3!")
print("\nModel locations:")
for model_key, model_data in model_info.items():
    print(f"  {model_key}: {model_data['s3_uri']}/model.tar.gz")

# Next Steps

You've successfully set up your environment, created the workshop configuration, and uploaded the base models to S3. The other notebooks can now use these models as inputs for their processing jobs.

In the next notebook, we'll explore quantization techniques to reduce model size while maintaining performance.